# Fashion-MNIST Capstone: Linear Baseline vs. CNN

**Runnable training notebook for Google Colab.** The project's source code is embedded in this notebook, so you do not need to clone GitHub or upload a second ZIP file. It trains a linear baseline and a CNN from scratch, evaluates them on the untouched official test set, saves learning curves and genuine prediction examples, and generates the PDF report from measured results.

## Before running
1. Download this `.ipynb` file from the ChatGPT conversation.
2. In Colab, use **File → Upload notebook** and select this file.
3. Choose **Runtime → Change runtime type → T4 GPU** if available.
4. Run all cells from the top. The notebook unpacks the embedded source, installs dependencies, downloads Fashion-MNIST, trains, evaluates, and creates outputs.

**Important:** real metrics and failure cases appear only after the training finishes successfully. Do not submit a draft report with pending placeholders.

In [ ]:
# Step 1 — Materialize the project source embedded in this notebook
# No additional project upload or GitHub clone is required.
import base64, zipfile, shutil, os
from pathlib import Path

PROJECT_BUNDLE_B64 = (
    '...'
)
bundle_path = Path('/content/fashion_mnist_runtime_source.zip')
bundle_path.write_bytes(base64.b64decode(PROJECT_BUNDLE_B64))
project_dir = Path('/content/fashion-mnist-capstone')
if project_dir.exists():
    shutil.rmtree(project_dir)
with zipfile.ZipFile(bundle_path) as zf:
    zf.extractall('/content')
if not (project_dir / 'src' / 'train.py').exists():
    raise FileNotFoundError('Embedded project source did not extract correctly.')
os.chdir(project_dir)
print('Project directory:', Path.cwd())
print('Source files loaded:', len(list(project_dir.rglob('*.py'))))
print('Ready to install dependencies and train.')

In [ ]:
# Step 2 — Install dependencies and confirm the accelerator
%pip -q install -r requirements.txt
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: no GPU detected. Training still runs on CPU, but will take longer.')

In [ ]:
# Step 3 — Run the project unit tests before training
!python -m pytest -q

## 1. Train both models

The protocol uses 55,000 training images, 5,000 validation images sampled deterministically from the official training partition, and the official 10,000-image test partition. The test set is not used for early stopping or model selection. Each model is initialized from scratch. Hyperparameters below are documented and reproducible with seed 42.

In [ ]:
# Step 4 — Actual training on Fashion-MNIST (this is the compute-heavy cell)
!python -m src.train --model all --epochs 12 --batch-size 128 --seed 42 --patience 3 --num-workers 2

## 2. Inspect measured test performance and learning curves

Accuracy and macro-F1 below are read from the JSON written by the evaluation script. The values are not hard-coded.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

metrics_path = Path('results/metrics.json')
if not metrics_path.exists():
    raise FileNotFoundError('No metrics file found. Training must finish successfully before evaluation can be reported.')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
rows = []
for item in metrics['models']:
    rows.append({
        'Model': 'Linear baseline' if item['model'] == 'linear' else 'Custom CNN',
        'Test accuracy': round(item['test_accuracy'], 4),
        'Test macro-F1': round(item['test_macro_f1'], 4),
        'Best validation loss': round(item['best_validation_loss'], 4),
        'Epochs completed': item['epochs_completed'],
        'Training time (s)': item.get('training_seconds', None),
    })
display(pd.DataFrame(rows))
print('Device recorded:', metrics.get('device'))
print('Seed:', metrics.get('seed'))
display(Image(filename='results/learning_curves.png'))

## 3. Confusion matrix and concrete success/failure cases

Use these actual held-out test examples in the report and demo. The failure examples are selected from genuine CNN errors (high-confidence mistakes first); inspect one and explain a plausible reason without claiming certainty.

In [ ]:
for artifact in ['results/confusion_matrix_cnn.png', 'results/success_cases.png', 'results/failure_cases.png']:
    path = Path(artifact)
    if path.exists():
        print(artifact)
        display(Image(filename=str(path)))
    else:
        print('Missing artifact:', artifact, '— check the training/evaluation output above.')

failure_json = Path('results/failure_cases.json')
if failure_json.exists():
    failures = json.loads(failure_json.read_text(encoding='utf-8'))
    if failures:
        print('Example of a genuine failure case:')
        print(json.dumps(failures[0], indent=2, ensure_ascii=False))
    else:
        print('No failure case was produced; verify the evaluator before completing the assignment.')

## 4. Generate the project report PDF from the experiment

The report generator includes measured accuracy/F1, the saved learning curves, and a real test-set failure case when the training artifacts exist.

In [ ]:
!python -m src.make_report --output report.pdf
from pathlib import Path
report_path = Path('report.pdf')
assert report_path.exists() and report_path.stat().st_size > 10_000, 'PDF report was not created correctly.'
print('Report created:', report_path.resolve(), f'({report_path.stat().st_size:,} bytes)')
from google.colab import files
files.download(str(report_path))

## 5. Download the experiment artifacts

This bundle contains the real metrics, plots, prediction examples, checkpoints and generated PDF. Keep it with the submission to support the numbers you report.

In [ ]:
# Create a compact artifact bundle for download (dataset files are intentionally excluded)
import zipfile
bundle = Path('fashion_mnist_experiment_artifacts.zip')
include = [Path('report.pdf'), Path('results'), Path('checkpoints')]
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for item in include:
        if item.is_file():
            zf.write(item, item.as_posix())
        elif item.is_dir():
            for file_path in item.rglob('*'):
                if file_path.is_file() and file_path.name != '.gitkeep':
                    zf.write(file_path, file_path.as_posix())
print('Artifact bundle:', bundle.resolve(), f'({bundle.stat().st_size:,} bytes)')
files.download(str(bundle))

## 6. Demo checklist (3–6 minutes)

- Show the project README and the command used to reproduce training.
- Explain the seeded train/validation/test split and why the test set stays untouched during tuning.
- Show the training/validation curves and the measured baseline-versus-CNN table.
- Show one correct prediction and one genuine misclassification; read the labels from the generated examples.
- Explain one limitation (for example, low resolution and visually similar clothing classes).

**Do not invent or edit result numbers.** If the notebook errors during dataset download or training, include that error when asking for help rather than submitting placeholder metrics.